In [ ]:
# Example 1: Create a NucleotideSequence and exercise basic sequence operations.
import biotite.sequence as seq

dna = seq.NucleotideSequence("ATGCGATTACAGTCA")
print(f"sequence: {dna}")
print(f"length: {len(dna)}")
print(f"complement: {dna.complement()}")
print(f"reverse complement: {dna.reverse().complement()}")

# GC content
gc = sum(1 for s in str(dna) if s in 'GC') / len(dna)
print(f"GC content: {gc:.3f}")

assert len(dna) == 15
assert str(dna.complement()) == "TACGCTAATGTCAGT"


In [ ]:
# Example 2: Translate a nucleotide sequence to protein.
import biotite.sequence as seq

# Construct an in-frame coding sequence (start + a few codons + stop).
coding = seq.NucleotideSequence("ATGGCCATTGTAATGGGCCGCTGAAAGGGTGCCCGATAG")
print(f"coding sequence: {coding}")

# Translate returns (list of ProteinSequence, list of (start, stop)).
proteins, positions = coding.translate()
print(f"translated {len(proteins)} ORF(s)")
for protein, pos in zip(proteins, positions):
    print(f"  ORF at {pos}: {protein}")

assert len(proteins) >= 1
assert isinstance(proteins[0], seq.ProteinSequence)


In [ ]:
# Example 3: Pairwise sequence alignment using a substitution matrix.
import biotite.sequence as seq
import biotite.sequence.align as align

seq1 = seq.ProteinSequence("BIQTITE")
seq2 = seq.ProteinSequence("TITANITE")
matrix = align.SubstitutionMatrix.std_protein_matrix()

alignments = align.align_optimal(seq1, seq2, matrix, gap_penalty=(-10, -1))
print(f"found {len(alignments)} optimal alignment(s)")
best = alignments[0]
print(f"score: {best.score}")
print(best)

assert best.score is not None
assert len(alignments) >= 1


In [ ]:
# Example 4: Build an AtomArray and exercise structure operations.
import numpy as np
import biotite.structure as struc

# Construct a 3-atom structure (a tiny water molecule).
atoms = struc.AtomArray(length=3)
atoms.coord = np.array([
    [0.0, 0.0, 0.0],
    [0.96, 0.0, 0.0],
    [-0.24, 0.93, 0.0],
])
atoms.element = np.array(["O", "H", "H"])
atoms.atom_name = np.array(["O", "H1", "H2"])
atoms.res_name = np.array(["HOH", "HOH", "HOH"])
atoms.res_id = np.array([1, 1, 1])
atoms.chain_id = np.array(["A", "A", "A"])

# Distance between the two H atoms
d = struc.distance(atoms[1], atoms[2])
print(f"H-H distance: {d:.3f} A")
print(f"array length: {len(atoms)}")

assert len(atoms) == 3
assert d > 0


In [ ]:
# Example 5: GeneralSequence and codon table introspection.
import biotite.sequence as seq

# Default standard codon table.
table = seq.CodonTable.default_table()
print(f"start codons: {table.start_codons()}")

# biotite 1.x removed CodonTable.stop_codons(); stop codons are the entries of
# codon_dict() that map to the stop symbol '*'.
stop_codons = tuple(codon for codon, aa in table.codon_dict().items() if aa == "*")
print(f"stop codons: {stop_codons}")

# Look up a few specific codons (getitem still returns the amino-acid symbol).
for codon in ["ATG", "TAA", "GCC", "TTT"]:
    aa = table[codon]
    print(f"  {codon} -> {aa}")

# Reverse lookup: which codons encode Leucine ('L')?
leucine_codons = table["L"]
print(f"leucine codons: {leucine_codons}")

assert "ATG" in table.start_codons()
assert "TAA" in stop_codons
assert len(leucine_codons) >= 1
print("\nall biotite smoke tests passed")